# Physical Axicon Beam Study

Stage C notebook for the physical axicon route. This route does not use a holographic blaze carrier or first-order selection in the same sense as the SLM hologram route; it has separate aperture, efficiency, alignment, and hardware assumptions.

In [1]:
from dataclasses import replace
from pathlib import Path

import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_regime, vbb_train_viz
from vbb_study.equations import objective_pupil as objp
from vbb_study.publication import lab_realism as lab_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
base = replace(bt.default_config(PRESET), generation_method="physical")
out_fig = PATHS["figures"] / "stage_c"
out_csv = PATHS["csv"] / "stage_c"
out_fig.mkdir(parents=True, exist_ok=True)
out_csv.mkdir(parents=True, exist_ok=True)

def _objective_fields(cfg):
    return {
        "objective_NA": float(cfg.objective.NA),
        "objective_f_eff_mm": float(cfg.objective.f_eff_m / bt.mm),
        "pupil_radius_mm": float(cfg.objective.pupil_radius_m / bt.mm),
        "pupil_clipped_fraction": objp.gaussian_clipping_power_fraction(
            cfg.laser.beam_radius_on_slm_m,
            cfg.objective.pupil_radius_m,
        ),
    }

def _hardware_status(path_label):
    return "future_hardware_required" if path_label == "lab" else "simulation_only"


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides to `base` before running any study cell below.
from vbb_study.publication import notebook_controls as nb_controls
from vbb_study.config import um as _um

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='lab_realism',
    # ── edit these to override the base configuration ───────────────────────
    ell=3,
    target_core_diameter_um=3.0,
    target_bessel_length_um=150.0,
    objective_NA=0.45,
    # blaze_period_px=20,  # holographic route only
)

# Wire control parameters into `base` so downstream cells use them.
_p = NOTEBOOK_CONTROLS.parameters or {}
if "ell" in _p:
    base = replace(base, target=replace(base.target, ell=int(_p["ell"])))
if "target_core_diameter_um" in _p:
    base = replace(base, target=replace(base.target, target_core_diameter_m=float(_p["target_core_diameter_um"]) * _um))
if "target_bessel_length_um" in _p:
    base = replace(base, target=replace(base.target, target_bessel_length_m=float(_p["target_bessel_length_um"]) * _um))
if "objective_NA" in _p:
    base = replace(base, objective=replace(base.objective, NA=float(_p["objective_NA"])))
if "blaze_period_px" in _p:
    base = replace(base, slm=replace(base.slm, blaze_period_px=int(_p["blaze_period_px"])))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


,control,value
0,stage,lab_realism
1,run_mode,balanced
2,save_outputs,False
3,use_canonical_outputs,True
4,allow_publication_export,False
5,notes,Edit for exploration; keep QA labels/caveats v...
6,ell,3
7,target_core_diameter_um,3.0
8,target_bessel_length_um,150.0
9,objective_NA,0.45


## Interactive Quicklook

Adjust sliders and click **Update plots** to explore parameter combinations instantly.
This cell runs a `preset='fast'` preview only — it does not write any saved outputs and does not affect the locked study cells below.

- **ell** — vortex charge (0 = scalar Bessel, ≥1 = vortex ring)
- **core diam** — equivalent J₀ first-zero target diameter
- **zone length** — target non-diffracting propagation length
- **objective NA** — changes demagnification and pupil size
- **SLM2 levels** — quantization of the SLM2 conjugate mask (0 = ideal continuous)


In [ ]:
# Interactive quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved.
from vbb_study.publication import notebook_widgets as nbw

_panel = nbw.interactive_quicklook(base, method='physical', preset='fast')
display(_panel)


In [ ]:
nb02_results = {}
rows = []
for regime in ("general", "limits"):
    cfg = vbb_regime.config_for_regime(base, regime)
    ideal_cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=None, slm2_conjugate_mode="full"))
    lab_cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=256, slm2_conjugate_mode="full"))
    for label, run_cfg in [("ideal", ideal_cfg), ("lab", lab_cfg)]:
        result = bt.run_case(run_cfg, preset=PRESET, path="ideal", case_id=f"{regime}_physical_{label}")
        nb02_results[(regime, label)] = result
        m = result["metrics"]
        meta = result["axicon_metadata"]
        design = bt.compute_design_from_targets(run_cfg.laser, run_cfg.target, run_cfg.material)
        row = {
            "case_id": f"{regime}_physical_{label}",
            "regime": regime,
            "path": "ideal",
            "route_variant": label,
            "physical_axicon_phase": meta.get("physical_axicon_phase"),
            "physical_axicon_pixelated": meta.get("physical_axicon_pixelated"),
            "physical_axicon_base_angle_deg": meta.get("gamma_deg", design.gamma_slm_deg),
            "equivalent_kr_m_inv": result["axicon_result"].k_r,
            "predicted_bessel_length_um": design.target_bessel_length_m / bt.um,
            "canonical_zone_um": m["canonical_zone_um"],
            "strict_bessel_region_um": m["strict_bessel_region_um"],
            "feature_diameter_um": m["feature_diameter_um"],
            "peak_fluence_J_cm2": m["peak_fluence_J_cm2"],
            "side_to_core_peak_ratio": m["side_to_core_peak_ratio"],
            "first_order_selected_fraction": m.get("first_order_selected_fraction"),
            "propagation_power_drift_fraction": m.get("propagation_power_drift_fraction"),
            "propagation_power_label": m.get("propagation_power_label"),
            "slm2_residual_phase_rms_before_rad": meta.get("slm2_residual_phase_rms_before_rad"),
            "slm2_residual_phase_rms_after_rad": meta.get("slm2_residual_phase_rms_after_rad"),
            "validity_valid": result["validity_report"]["valid"],
            **_objective_fields(run_cfg),
        }
        lab_schema.annotate_lab_realism_row(
            row,
            generation_method="physical_axicon",
            model_level="hardware_route",
            hardware_status=_hardware_status(label),
            plane_label="surface_plane",
            coordinate_frame="surface_plane_air_um",
            run_id=RUN_ID,
            preset=PRESET,
            path="ideal",
        )
        rows.append(row)
summary = lab_schema.ordered_lab_realism_frame(rows)
summary.to_csv(out_csv / "physical_axicon_design_summary.csv", index=False)
summary

In [ ]:
from vbb_study.viz_fields import measured_charge_label as _mcl_nb02
_sf_phys = nb02_results[("general", "lab")]["surface_field"]
_des_phys = nb02_results[("general", "lab")]["design"]
_phys_charge_lbl = _mcl_nb02(
    _sf_phys.Ex, _sf_phys.grid,
    float(_des_phys.vortex_main_ring_radius_m),
    design_ell=int(_des_phys.ell), conjugate_mode="full",
)
print(f"Physical charge label (general/lab): {_phys_charge_lbl}")
vbb_train_viz.plot_train_visualiser(base, method='physical', output_dir=out_fig, charge_label=_phys_charge_lbl)
vbb_train_viz.plot_sampling_qa(base, output_dir=out_fig)

## Hero Field Views at Balanced Preset (N=1024)

Physical route lab config — `linked_field_views` (4-panel) and `azimuthal_order_panel` at `balanced` preset (N=1024, device_downsample=2) for both regimes.

**Finding F-A3p reminder:** `slm2_conjugate_mode='full'` strips the SLM1 helical phase before the axicon, so the physical beam has winding ≈ 0 regardless of design ℓ. The charge label below is MEASURED from the field, not hardcoded.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from vbb_study.viz_fields import linked_field_views, azimuthal_order_panel, measured_charge_label

HERO_PRESET = "balanced"
out_fig.mkdir(parents=True, exist_ok=True)
for regime in ("general", "limits"):
    cfg = vbb_regime.config_for_regime(base, regime)
    hero_cfg = replace(cfg, physical_axicon=replace(
        cfg.physical_axicon, slm2_stroke_levels=256, slm2_conjugate_mode="full"
    ))
    hero_result = bt.run_case(
        hero_cfg, preset=HERO_PRESET, path="ideal",
        case_id=f"{regime}_physical_lab_hero",
    )
    sf = hero_result["surface_field"]
    design = hero_result["design"]
    sample_r_m = float(design.vortex_main_ring_radius_m)
    N_hero = hero_cfg.grid.N
    ds_hero = hero_cfg.grid.device_downsample
    charge_lbl = measured_charge_label(
        sf.Ex, sf.grid, sample_r_m,
        design_ell=int(design.ell), conjugate_mode="full",
    )
    caption = (
        f"Physical | {regime} | {charge_lbl} | "
        f"preset={HERO_PRESET}, N={N_hero}, device_downsample={ds_hero}"
    )
    fig, axes = linked_field_views(hero_result, title_prefix=f"Physical {regime}")
    fig.suptitle(caption, fontsize=8, wrap=True)
    fig.savefig(
        out_fig / f"nb02_physical_{regime}_hero_linked_field_views.png",
        dpi=150, bbox_inches="tight",
    )
    plt.close(fig)

    # Azimuthal order panel — ring sample from surface-plane field
    n_phi_ring = 512
    x_arr = np.asarray(sf.grid["x"], dtype=float)
    dx_sf = float(sf.grid["dx"])
    x0_sf = float(x_arr[0])
    phis_ring = np.linspace(0.0, 2.0 * np.pi, n_phi_ring, endpoint=False)
    col_r = (sample_r_m * np.cos(phis_ring) - x0_sf) / dx_sf
    row_r = (sample_r_m * np.sin(phis_ring) - x0_sf) / dx_sf
    I_ring = map_coordinates(np.abs(sf.Ex) ** 2, [row_r, col_r], order=1, mode="nearest")
    fig2, _ = azimuthal_order_panel(
        I_ring, ell=int(design.ell),
        title=f"Physical {regime} azimuthal order | {charge_lbl}",
    )
    fig2.savefig(
        out_fig / f"nb02_physical_{regime}_azimuthal_order.png",
        dpi=150, bbox_inches="tight",
    )
    plt.close(fig2)
    print(f"  {regime}: {charge_lbl}")